# 10. テーブルフォーマット v3 へのアップグレード（追加目標）

このハンズオンのテーブルはすべて v2 です。v3 では主に次の機能が加わりました。

| 機能 | 内容 |
| --- | --- |
| 削除ベクトル | 削除した行を、データファイルごとのビットマップ（Puffin 形式のファイル）で記録する。v2 の位置削除ファイルより小さく、読むときも速い |
| 行の履歴（row lineage） | 各行に `_row_id` と `_last_updated_sequence_number` が付き、行がいつ変更されたかを追える |
| 新しい型 | variant（半構造化データ）、ナノ秒のタイムスタンプ、地理空間型など |

v2 のテーブルを v3 にアップグレードし、削除の記録のされ方と行の履歴を確かめます。
**アップグレードは一方通行で、v2 には戻せません。**

- テーブル: `handson.v3_spark`
- エンジンごとの v3 対応状況は `docs/design.md` を参照（Trino は実験的な対応、PyIceberg はアップグレード不可）

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("10_v3_upgrade").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 1. v2 のテーブルで DELETE する

DELETE は削除ファイルを書き足す方式（merge-on-read）にしておきます。
v2 では、削除は **位置削除ファイル（Parquet）** として記録されます（`content = 1`）。データファイルはそのままです。

In [ ]:
sql("DROP TABLE IF EXISTS handson.v3_spark PURGE")
sql("""
CREATE TABLE handson.v3_spark (trip_id BIGINT, vendor STRING, fare DECIMAL(10, 2))
USING iceberg
TBLPROPERTIES ('write.delete.mode' = 'merge-on-read', 'write.update.mode' = 'merge-on-read')
""")
# 削除ファイルの働きが見えるよう、4 行を 1 つのファイルにまとめて書く（REPARTITION(1)）
sql("""
INSERT INTO handson.v3_spark
SELECT /*+ REPARTITION(1) */ * FROM VALUES (1, 'A', 12.50), (2, 'B', 30.00), (3, 'A', 8.00), (4, 'C', 22.00)
""")
sql("DELETE FROM handson.v3_spark WHERE trip_id = 2")

def show_files():
    sql("SELECT content, file_format, record_count, file_size_in_bytes FROM handson.v3_spark.files ORDER BY content")

sql("SHOW TBLPROPERTIES handson.v3_spark ('format-version')")
show_files()

## 2. v3 にアップグレードする

テーブルのプロパティ `format-version` を 3 にするだけです。データファイルは書き直されません。

In [ ]:
sql("ALTER TABLE lakehouse.handson.v3_spark SET TBLPROPERTIES ('format-version' = '3')")
sql("SHOW TBLPROPERTIES handson.v3_spark ('format-version')")

## 3. v3 のテーブルで DELETE する

同じ DELETE でも、今度は **削除ベクトル（Puffin 形式）** として記録されます。
v3 では、1 つのデータファイルに対する削除は 1 つの削除ベクトルにまとめられます。

In [ ]:
sql("DELETE FROM handson.v3_spark WHERE trip_id = 3")
show_files()

## 4. 行の履歴（row lineage）

v3 のテーブルでは、各行に隠し列 `_row_id`（行の ID）と `_last_updated_sequence_number`（最後に変更されたときのシーケンス番号）があります。
UPDATE の前後で比べます。

In [ ]:
print("UPDATE の前")
sql("SELECT trip_id, fare, _row_id, _last_updated_sequence_number FROM handson.v3_spark ORDER BY trip_id")

sql("UPDATE handson.v3_spark SET fare = 13.00 WHERE trip_id = 1")

print("UPDATE の後")
sql("SELECT trip_id, fare, _row_id, _last_updated_sequence_number FROM handson.v3_spark ORDER BY trip_id")

UPDATE した行だけ `_last_updated_sequence_number` が新しくなっています。
これを使うと、「前回の処理以降に変更された行だけを取り出す」といった差分処理ができます。

## 5. 他のエンジンから読む

PyIceberg からも v3 のテーブルを読めます（PyIceberg 0.12 はアップグレードの操作はできませんが、読み書きはできます）。
Trino からの読み書きは trino.sql で試します。

In [ ]:
from pyiceberg.catalog import load_catalog

table = load_catalog("lakehouse").load_table("handson.v3_spark")
print("format-version:", table.metadata.format_version)
table.scan().to_pandas().sort_values("trip_id")

## まとめ

- `format-version` を 3 にするだけで、データを書き直さずにアップグレードできる（元には戻せない）
- v3 では削除が削除ベクトル（Puffin）として記録される
- v3 では各行に `_row_id` と `_last_updated_sequence_number` が付き、行の変更を追える
- エンジンによって v3 への対応状況が違うので、アップグレードの前に、使うエンジンがすべて対応しているかを確かめる